файлы

In [1]:
import os
os.getcwd()

'/content'

In [2]:
os.listdir()


['.config',
 'sleep.csv',
 'physical.csv',
 'drive',
 'mental.csv',
 'noise.csv',
 'sample_data']

загрузка файлов

In [3]:
import pandas as pd
import numpy as np

files = {
    "sleep.csv": 1,
    "mental.csv": 2,
    "physical.csv": 3,
    "noise.csv": 0
}

for fn in files:
  df = pd.read_csv(fn)
  print(fn, df.columns.tolist(), 'len=', len(df))

sleep.csv ['signal'] len= 35016
mental.csv ['signal'] len= 52588
physical.csv ['signal'] len= 13678
noise.csv ['signal'] len= 35016


параметры окон

In [4]:
FS = 128
WINDOW_SEC = 6
STEP_SEC = 3

WINDOW = FS * WINDOW_SEC
STEP = FS * STEP_SEC

windowing по каждому файлу

In [5]:
def make_windows_1d(signal, labels, window, step):
  x = []
  y = []

  for i in range(0, len(signal)-window, step):
    x.append(signal[i:i+window])
    y.append(labels)

  return np.array(x), np.array(y)

x_all, y_all = [], []

for fn, labels in files.items():
  signal = pd.read_csv(fn)['signal'].to_numpy().astype(np.float32)
  x, y = make_windows_1d(signal, labels, WINDOW, STEP)
  x_all.append(x); y_all.append(y)

x = np.concatenate(x_all)
y = np.concatenate(y_all)

print('x shape:', x.shape)
print('y shape:', y.shape)
print('class counts:', np.bincount(y))

x shape: (349, 768)
y shape: (349,)
class counts: [ 90  90 135  34]


нормализация окон

In [6]:
x = x - x.mean(axis=1, keepdims=True)
x = x / (x.std(axis=1, keepdims=True) + 1e-8)

извлечение признаков

In [7]:
from scipy.fft import rfft, rfftfreq
import numpy as np

def extract_features(window, fs=128):
    w = window - window.mean()

    fft_vals = np.abs(rfft(w))
    freqs = rfftfreq(len(w), 1/fs)

    def band(lo, hi):
        idx = (freqs >= lo) & (freqs <= hi)
        return fft_vals[idx].mean() if np.any(idx) else 0.0

    # спектральные признаки
    delta = band(1, 4)
    theta = band(4, 8)
    alpha = band(8, 13)
    beta  = band(13, 30)

    # отношения диапазонов (ОЧЕНЬ важно для ЭЭГ)
    ratio_dt = delta / (theta + 1e-6)
    ratio_ab = alpha / (beta + 1e-6)
    ratio_ta = theta / (alpha + 1e-6)

    # временные признаки
    rms = np.sqrt(np.mean(w**2))
    zcr = np.mean(np.diff(np.sign(w)) != 0)

    return np.array([
        delta, theta, alpha, beta,
        ratio_dt, ratio_ab, ratio_ta,
        rms, zcr
    ], dtype=np.float32)


In [8]:
x_feat = np.stack([extract_features(win, fs=FS) for win in x])

# нормализация
x_feat = (x_feat - x_feat.mean(axis=0)) / (x_feat.std(axis=0) + 1e-8)

print("x_feat shape:", x_feat.shape)  # должно быть (N, 9)

x_feat shape: (349, 9)


разделение выборки на train/test

In [9]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(
    x_feat, y,
    test_size = 0.25,
    random_state = 42,
    stratify = y
)

print('train:', x_train.shape, 'test:', x_test.shape)

train: (261, 9) test: (88, 9)


In [10]:
import torch
import numpy as np
import torch.nn as nn

counts = np.bincount(y_train, minlength=4)
weights = 1.0 / torch.tensor(counts, dtype=torch.float32)
weights = weights / weights.sum() * 4  # нормировка, не критично, но удобно

print("train counts:", counts)
print("class weights:", weights)

criterion = nn.CrossEntropyLoss(weight=weights)

train counts: [ 67  67 101  26]
class weights: tensor([0.7633, 0.7633, 0.5064, 1.9670])


torch tensors

In [11]:
import torch
import torch.nn as nn

x_train_t = torch.tensor(x_train, dtype=torch.float32)
x_test_t  = torch.tensor(x_test, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)
y_test_t  = torch.tensor(y_test, dtype=torch.long)

модель mlp

In [12]:
class SimpleEEGNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(9, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 4)
        )

    def forward(self, x):
        return self.net(x)

model = SimpleEEGNet()

обучение

In [13]:
from torch.utils.data import TensorDataset, DataLoader

train_ds = TensorDataset(x_train_t, y_train_t)
train_dl = DataLoader(train_ds, batch_size=64, shuffle=True)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for epoch in range(30):
    model.train()
    total_loss = 0.0

    for xb, yb in train_dl:
        optimizer.zero_grad()
        out = model(xb)
        loss = criterion(out, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * xb.size(0)

    if epoch % 5 == 0:
        print(f"Epoch {epoch}, loss = {total_loss/len(train_ds):.4f}")

Epoch 0, loss = 1.3951
Epoch 5, loss = 1.3796
Epoch 10, loss = 1.3681
Epoch 15, loss = 1.3614
Epoch 20, loss = 1.3524
Epoch 25, loss = 1.3471


In [14]:
import numpy as np
from sklearn.metrics import confusion_matrix

# 1) accuracy на train и test
model.eval()
with torch.no_grad():
    pred_train = torch.argmax(model(x_train_t), dim=1).cpu().numpy()
    pred_test  = torch.argmax(model(x_test_t),  dim=1).cpu().numpy()

train_acc = (pred_train == y_train).mean()
test_acc  = (pred_test  == y_test ).mean()

print("Train acc:", train_acc)
print("Test acc :", test_acc)

# 2) распределение предсказаний на test (сколько раз модель выбрала каждый класс)
print("Pred counts (test):", np.bincount(pred_test, minlength=4))
print("True counts (test):", np.bincount(y_test, minlength=4))

# 3) confusion matrix
cm = confusion_matrix(y_test, pred_test, labels=[0,1,2,3])
print("Confusion matrix:\n", cm)

Train acc: 0.37547892720306514
Test acc : 0.2840909090909091
Pred counts (test): [ 2 24 41 21]
True counts (test): [23 23 34  8]
Confusion matrix:
 [[ 0  8 12  3]
 [ 0  6  8  9]
 [ 2 10 16  6]
 [ 0  0  5  3]]


метрики

In [15]:
from sklearn.metrics import confusion_matrix, classification_report

model.eval()
with torch.no_grad():
    logits = model(x_test_t)
    pred = torch.argmax(logits, dim=1).cpu().numpy()

acc = (pred == y_test).mean()
print("Accuracy:", acc)

print("Confusion matrix:\n", confusion_matrix(y_test, pred))

print("\nReport:\n", classification_report(y_test, pred, digits=3))

Accuracy: 0.2840909090909091
Confusion matrix:
 [[ 0  8 12  3]
 [ 0  6  8  9]
 [ 2 10 16  6]
 [ 0  0  5  3]]

Report:
               precision    recall  f1-score   support

           0      0.000     0.000     0.000        23
           1      0.250     0.261     0.255        23
           2      0.390     0.471     0.427        34
           3      0.143     0.375     0.207         8

    accuracy                          0.284        88
   macro avg      0.196     0.277     0.222        88
weighted avg      0.229     0.284     0.250        88



In [16]:
import numpy as np

print("x:", x.shape, "y:", y.shape)
print("unique labels:", np.unique(y))
print("class counts:", np.bincount(y))

# как выглядит случайный и "всегда один класс"
majority = np.argmax(np.bincount(y))
majority_acc = (y == majority).mean()
print("majority class:", majority, "majority baseline acc:", majority_acc)

# проверка, что нет NaN/inf после нормализации
print("x nan:", np.isnan(x).any(), "x inf:", np.isinf(x).any())
print("x mean (first 3 windows):", x[:3].mean(axis=1))
print("x std  (first 3 windows):", x[:3].std(axis=1))

x: (349, 768) y: (349,)
unique labels: [0 1 2 3]
class counts: [ 90  90 135  34]
majority class: 2 majority baseline acc: 0.3868194842406877
x nan: False x inf: False
x mean (first 3 windows): [ 4.0978193e-08  1.6142925e-08 -4.4703484e-08]
x std  (first 3 windows): [0.9999999 1.        1.       ]
